# Python Automation & ETL Basics: The Messy Grocery List
**Date:** 07/09/2026

This notebook demonstrates a basic ETL (Extract, Transform, Load) pipeline. 
We will take a messy`.txt` grocery list, clean the data using `pandas`, and store the final output in an Excel spreadsheet.


In [1]:
!pip install openpyxl
!mamba install pandas

mambajs 0.21.4

Process pip requirements ...

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 1.635 seconds
  Name           Version  Build                Channel
--------------------------------------------------------------------
+ pandas         3.0.4    np23py313h1e705a5_0  emscripten-forge-4x
+ python-tzdata  2026.2   pyhd8ed1ab_0         conda-forge
- pip            26.1.2   pyh145f28c_0         conda-forge


In [2]:
# Import necessary libraries
import pandas as pd # extract, transform
import logging
import os
import sys

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True,
    handlers=[
        logging.FileHandler("etl_pipeline.log", mode='w'),
        logging.StreamHandler()                  
    ]
)

## Step 1: Extract
Fetching raw data from our source file. We will read the `messy_grocery_list.txt` file into a pandas DataFrame so we can easily manipulate it.

In [4]:
def extract_data(filepath):
    """Reads a text file and loads it into a pandas DataFrame."""
    df = pd.read_csv(filepath, sep="-", header=None, names=["Item_Raw","Details_Raw"])
    return df

In [5]:
file_path = 'messy_grocery_list.txt'
raw_df = extract_data(file_path)
logging.info(f"Extraction complete. {len(raw_df)} rows loaded.")
raw_df.head()

2026-07-09 16:39:13,353 - INFO - Extraction complete. 5 rows loaded.


,Item_Raw,Details_Raw
0,apples,1 pieces
1,MILK!!!,2 cartons
2,banana,6 yellow banana
3,juicE,1 liter
4,MILK!!!,2 cartons


## Step 2: Transform
This is where `pandas` shines. We will clean the data by:
1. Cleaning the item names (removing special characters like `!!!`, standardizing to lowercase, stripping extra spaces, and removing duplicates).
2. Extracting the numerical value (quantity) and the unit of measurement into separate columns.

In [6]:
def transform_data(df):
    """Cleans and structures the raw dataframe."""
    logging.info("Starting data transformation...")
    try:
        logging.info(df[['Item_Raw', 'Details_Raw']].head())
        
        # 1. Clean the 'Item' column
        # Convert to lowercase, remove everything except letters (a-z), and strip edge spaces
        df['Item'] = df['Item_Raw'].str.lower().str.replace(r'[^a-z\s]', '', regex=True).str.strip()
        
        # 2. Clean the 'Details' column to extract Quantity and Unit
        # Extract the first number found as Quantity
        df['Quantity'] = df['Details_Raw'].str.extract(r'(\d+)').astype(float)
        
        # Extract the alphabetical characters after the number as the Unit
        df['Unit'] = df['Details_Raw'].str.replace(r'\d+', '', regex=True).str.strip()
        
        # 3. Filter down to only the clean columns we want
        clean_df = df[['Item', 'Quantity', 'Unit']].copy()

        # 4. Remove duplicated rows
        clean_df = clean_df.drop_duplicates(ignore_index=True)
        
        logging.info("Data transformation successful.")
        return clean_df
        
    except Exception as e:
        # Error handling allows the script to fail gracefully
        logging.error(f"An error occurred during transformation: {e}")
        raise

In [7]:
# Execute Transform
clean_df = transform_data(raw_df)
clean_df.head()

2026-07-09 16:39:13,420 - INFO - Starting data transformation...
2026-07-09 16:39:13,428 - INFO -        Item_Raw       Details_Raw
0       apples           1 pieces
1      MILK!!!          2 cartons
2       banana    6 yellow banana
3        juicE            1 liter
4      MILK!!!          2 cartons
2026-07-09 16:39:13,449 - INFO - Data transformation successful.


,Item,Quantity,Unit
0,apples,1.0,pieces
1,milk,2.0,cartons
2,banana,6.0,yellow banana
3,juice,1.0,liter


## Step 3: Load
Storing the processed data into its final destination. We will save this to a **clean Excel spreadsheet**.

In [8]:
def load_data(df, excel_filename):
    """Saves the cleaned dataframe to Excel."""
    logging.info("Starting load process...")
    try:
        # --- LOAD TO EXCEL ---
        df.to_excel(excel_filename, index=False, engine='openpyxl')
        logging.info(f"Successfully loaded data into Excel: {excel_filename}")
        
    except Exception as e:
        logging.error(f"Failed to load data: {e}")
        raise

In [9]:
# Execute Load
clean_df = transform_data(raw_df)
excel_file = 'cleaned_grocery_list.xlsx'

load_data(clean_df, excel_file)

2026-07-09 16:39:13,494 - INFO - Starting data transformation...
2026-07-09 16:39:13,496 - INFO -        Item_Raw       Details_Raw
0       apples           1 pieces
1      MILK!!!          2 cartons
2       banana    6 yellow banana
3        juicE            1 liter
4      MILK!!!          2 cartons
2026-07-09 16:39:13,504 - INFO - Data transformation successful.
2026-07-09 16:39:13,504 - INFO - Starting load process...
2026-07-09 16:39:14,284 - INFO - Successfully loaded data into Excel: cleaned_grocery_list.xlsx


In [10]:
logging.info("Verifying Excel load...")

try:
    df = pd.read_excel("cleaned_grocery_list.xlsx", index_col=None)
    logging.info("Pipeline executed successfully! Here is the final loaded data:")
    
except Exception as e:
    logging.error(f"Verification failed: {e}")

2026-07-09 16:39:14,304 - INFO - Verifying Excel load...
2026-07-09 16:39:14,355 - INFO - Pipeline executed successfully! Here is the final loaded data:


In [11]:
df.head()

,Item,Quantity,Unit
0,apples,1,pieces
1,milk,2,cartons
2,banana,6,yellow banana
3,juice,1,liter


In [12]:
# 1. Add a Price column 
# (In a real scenario, we might extract this from a database, but for the demo we'll hardcode it)
df['Price'] = [10.00, 50.50, 12.30, 24.75]

# 2. Transform: Multiply Quantity by Price to create the 'Amount' column
df['Amount'] = df['Quantity'] * df['Price']

# 3. Calculate the Total Amount
total_amount = df['Amount'].sum()

# We create a new row with the total and append it.
total_row = pd.DataFrame({'Item': ['TOTAL'], 'Quantity': [''], 'Unit': [''], 'Price': [''], 'Amount': [total_amount]})
df = pd.concat([df, total_row], ignore_index=True)

# 4. Load: Export the final, fully calculated table to Excel
df.to_excel("updated_grocery_list.xlsx", index=False)
logging.info("Updated grocery list with prices exported to Excel.")

2026-07-09 16:39:14,468 - INFO - Updated grocery list with prices exported to Excel.


In [13]:
# Display the final dataframe in Jupyter
df

,Item,Quantity,Unit,Price,Amount
0,apples,1,pieces,10.0,10.00
1,milk,2,cartons,50.5,101.00
2,banana,6,yellow banana,12.3,73.80
3,juice,1,liter,24.75,24.75
4,TOTAL,,,,209.55


In [14]:
logging.info(f"The total grocery bill is: Php{total_amount:.2f}")

2026-07-09 16:39:14,518 - INFO - The total grocery bill is: Php209.55


In [15]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True,
    handlers=[
        logging.FileHandler("etl_pipeline.log", mode='w'),                
    ]
)

# Why print() is not enough
The print() output disappears once the terminal/console closes.

Logging creates a permanent and searchable history or record of
events during script execution, helping users to quickly identify,
debug, and fix issues by pinpointing the root cause of errors or
failures.